
# Lecture 5 — Biopython: Hands-on Exercises

> This notebook accompanies **Lecture 5** and covers: `Seq` / `SeqRecord`, `SeqIO` / `AlignIO`, wrappers for external aligners, BLAST (`NCBIWWW.qblast` and `NCBIXML`), Entrez E-utilities, motifs (PCM→PWM→PSSM), and simple phylogenetics with `Bio.Phylo`.

**How to use this notebook**
- Each exercise includes a **Task** cell you should run/edit.
- The **Answer** is hidden; click **“View answer”** to reveal it.
- Cells that require internet access (BLAST/Entrez) are marked **(online)**. You can still study the workflow offline.


In [ ]:

# Imports used across exercises
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO, AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.Align import substitution_matrices
from Bio.Align.Applications import ClustalwCommandline, MuscleCommandline, TCoffeeCommandline
from Bio.SeqUtils import GC
from Bio import Phylo
from Bio import motifs
from io import StringIO
import textwrap
print("Imports ready.")



## 1) `Seq` basics — reverse complement, translation, GC

**Task:**  
Given the DNA sequence `ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG`, do the following:

1. Construct a `Seq` object.
2. Compute the **reverse complement**.
3. Translate it to amino acids **stopping at the first stop codon**.
4. Compute **GC%**.
5. Explain briefly why `Seq` behaves like a Python string but adds biology-aware methods.

> Use: `Seq(...).reverse_complement()`, `.translate(to_stop=True)`, and `SeqUtils.GC()`.


In [ ]:

# TODO: Your work here
dna = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"
# 1) make Seq
# 2) reverse complement
# 3) translation to_stop=True
# 4) GC%
# 5) short explanation as a Python comment



<details><summary><strong>View answer</strong></summary>

```python
dna = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"
s = Seq(dna)
revcomp = s.reverse_complement()
protein = s.translate(to_stop=True)
gc = GC(s)

print("Seq:", s)
print("RevComp:", revcomp)
print("Protein (to first stop):", protein)
print("GC%:", round(gc, 2))

# Explanation:
# Seq acts like a Python string (supports slicing, len, etc.) but provides domain-specific
# methods like reverse_complement(), transcribe(), translate(), back_transcribe(), etc.
```
</details>



## 2) `SeqRecord` & `SeqIO` — parse FASTA and compute lengths

**Task:**  
Use the small in-memory FASTA below. Parse it into `SeqRecord` objects, print each record's `id`, description, first 10 bases, and length. Then collect lengths into a list.

FASTA:
```
>seq1 hypothetical protein
ATGCGTATCGATCGATCGATCGTAG
>seq2 kinase-like domain
ATGGGCCCCTTTAAAATTTGGGCCCAAATGA
```

> Hint: `SeqIO.parse(StringIO(fasta_str), "fasta")`


In [ ]:

# TODO: Your work here
fasta_str = ">seq1 hypothetical protein\nATGCGTATCGATCGATCGATCGTAG\n>seq2 kinase-like domain\nATGGGCCCCTTTAAAATTTGGGCCCAAATGA\n"
# Parse with SeqIO.parse and print required fields



<details><summary><strong>View answer</strong></summary>

```python
from io import StringIO
handle = StringIO(fasta_str)
lengths = []
for rec in SeqIO.parse(handle, "fasta"):
    print(rec.id, "|", rec.description, "| first10:", str(rec.seq)[:10], "| len:", len(rec))
    lengths.append(len(rec))
lengths
```
</details>



## 3) `AlignIO` — read a small alignment and slice columns

**Task:**  
Read the following minimal **Clustal** alignment string using `AlignIO.read(..., "clustal")`. Then:

1. Print number of sequences and alignment length.
2. Extract the **3rd column** across all sequences (index `2`) and print it as a string.
3. Slice the **first 5 columns** to create a sub-alignment.

Alignment (Clustal format):
```
CLUSTAL W (1.82) multiple sequence alignment

sp|A     ATGCGTATCG
sp|B     ATG-GTATCA
sp|C     ATACGTATCA
```

> Hint: You may need to ensure lines wrap correctly; we already formatted with a blank line after the header, as typical for Clustal output.


In [ ]:

# TODO: Your work here
clustal_str = textwrap.dedent("""CLUSTAL W (1.82) multiple sequence alignment

sp|A     ATGCGTATCG
sp|B     ATG-GTATCA
sp|C     ATACGTATCA
""")
# Read with AlignIO.read(StringIO(...), "clustal")
# Then do the operations requested



<details><summary><strong>View answer</strong></summary>

```python
aln = AlignIO.read(StringIO(clustal_str), "clustal")
print("Num seqs:", len(aln))
print("Alignment length:", aln.get_alignment_length())

col3 = aln[:, 2]  # 3rd column (0-based)
print("3rd column:", col3)

sub = aln[:, :5]
print(sub)
```
</details>



## 4) Command-line wrappers — ClustalW / MUSCLE / T-Coffee (offline)

**Task:**  
Write code that **prepares** a `ClustalwCommandline` run on a file `"example.fasta"` producing `"example.aln"` and `"example.dnd"`. **Do not execute** it here. Then write equivalent commandlines for **MUSCLE** and **T-Coffee**.

> Use classes from `Bio.Align.Applications`. On many systems, you'd need the external binaries installed and in PATH.


In [ ]:

# TODO: Your work here (do not run the commandline)
# clustalw_cline = ClustalwCommandline("clustalw2", infile="example.fasta")
# muscle_cline = MuscleCommandline(input="example.fasta", out="example_muscle.aln")
# tcoffee_cline = TCoffeeCommandline(infile="example.fasta", output="clustalw", outfile="example_tcoffee.aln")
# print(clustalw_cline)
# print(muscle_cline)
# print(tcoffee_cline)



<details><summary><strong>View answer</strong></summary>

```python
clustalw_cline = ClustalwCommandline("clustalw2", infile="example.fasta")
print(clustalw_cline)  # would produce example.aln and example.dnd

muscle_cline = MuscleCommandline(input="example.fasta", out="example_muscle.aln")
print(muscle_cline)

tcoffee_cline = TCoffeeCommandline(infile="example.fasta", output="clustalw", outfile="example_tcoffee.aln")
print(tcoffee_cline)
```
</details>



## 5) BLAST via `NCBIWWW.qblast` (online)

**Task (workflow):**  
Outline code that would:
1. Run `qblast("blastn", "nt", "ATGCGTATCG")`.
2. Save the XML to `blast_result.xml`.
3. Parse the XML with `NCBIXML.read` and print `hit.title`, `hsp.expect`, and the alignment snippet for all HSPs with **E-value < 0.001**.

> Do **not** execute online requests in class. Show the code only.


In [ ]:

# TODO: Outline only; do not execute.
# from Bio.Blast import NCBIWWW, NCBIXML
# result_handle = NCBIWWW.qblast("blastn", "nt", "ATGCGTATCG")
# with open("blast_result.xml", "w") as out:
#     out.write(result_handle.read())
# result_handle.close()
#
# with open("blast_result.xml") as inp:
#     blast_record = NCBIXML.read(inp)
# e_thresh = 1e-3
# for alignment in blast_record.alignments:
#     for hsp in alignment.hsps:
#         if hsp.expect < e_thresh:
#             print(alignment.title, hsp.expect)
#             print(hsp.query[0:60] + "...")
#             print(hsp.match[0:60] + "...")
#             print(hsp.sbjct[0:60] + "...")



<details><summary><strong>View answer</strong></summary>

```python
from Bio.Blast import NCBIWWW, NCBIXML
# 1) Run BLAST
# result_handle = NCBIWWW.qblast("blastn", "nt", "ATGCGTATCG")
# 2) Save XML
# with open("blast_result.xml", "w") as out:
#     out.write(result_handle.read())
# result_handle.close()

# 3) Parse & filter
# with open("blast_result.xml") as inp:
#     blast_record = NCBIXML.read(inp)
# e_thresh = 1e-3
# for alignment in blast_record.alignments:
#     for hsp in alignment.hsps:
#         if hsp.expect < e_thresh:
#             print(alignment.title, hsp.expect)
#             print(hsp.query[0:60] + "...")
#             print(hsp.match[0:60] + "...")
#             print(hsp.sbjct[0:60] + "...")
```
</details>



## 6) Entrez E-utilities (online)

**Task (workflow):**  
Write code to fetch a GenBank record **in text mode** for accession `"J01673"` (human insulin), printing its first 25 lines. Remember to set your email.

> Do **not** execute in class without network access.


In [ ]:

# TODO: Outline only; do not execute.
# from Bio import Entrez
# Entrez.email = "your_email@example.com"
# with Entrez.efetch(db="nucleotide", id="J01673", rettype="gb", retmode="text") as handle:
#     gb_txt = handle.read()
# print("\n".join(gb_txt.splitlines()[:25]))



<details><summary><strong>View answer</strong></summary>

```python
from Bio import Entrez
Entrez.email = "your_email@example.com"
with Entrez.efetch(db="nucleotide", id="J01673", rettype="gb", retmode="text") as handle:
    gb_txt = handle.read()
print("\n".join(gb_txt.splitlines()[:25]))
```
</details>



## 7) Motifs — build PCM→PWM→PSSM and scan

**Task:**  
Given the following motif instances, build a motif with `Bio.motifs`, compute the **consensus** and **anticonsensus**, derive its **PWM** and **PSSM** (log-odds), and scan the test sequence for positions with PSSM score ≥ 5.

Instances:
```
TACGC
TATGC
TACGC
TACAC
TACGC
```

Test sequence: `TTTACGCTAGTACGCGGTACGCT`

> Hints: `motifs.create([...])`, `m.counts.normalize(pseudocounts=0.5)`, `m.pwm.log_odds()`


In [ ]:

# TODO: Your work here
instances = [Seq(s) for s in ["TACGC","TATGC","TACGC","TACAC","TACGC"]]
# Build motif, print consensus/anticonsensus, derive PWM/PSSM, scan test sequence



<details><summary><strong>View answer</strong></summary>

```python
m = motifs.create(instances)
print("Consensus:", m.consensus)
print("Anticonsensus:", m.anticonsensus)

# PWM with pseudocounts & background correction (uniform background assumed)
pwm = m.counts.normalize(pseudocounts=0.5)
pssm = pwm.log_odds()

seq = Seq("TTTACGCTAGTACGCGGTACGCT")
hits = []
for i in range(len(seq) - len(m.consensus) + 1):
    window = seq[i:i+len(m.consensus)]
    score = pssm.calculate(window)
    if score >= 5:
        hits.append((i, str(window), float(score)))
hits
```
</details>



## 8) `Bio.Phylo` — read a small Newick tree and draw ASCII

**Task:**  
Given this Newick string, parse it and draw an ASCII tree.

Newick:
```
((seq1:0.1,seq2:0.2):0.3,(seq3:0.2,seq4:0.1):0.4);
```

> Hints: `Phylo.read(StringIO(newick), "newick")`, `Phylo.draw_ascii(tree)`


In [ ]:

# TODO: Your work here
newick = "((seq1:0.1,seq2:0.2):0.3,(seq3:0.2,seq4:0.1):0.4);"
# Parse and draw_ascii



<details><summary><strong>View answer</strong></summary>

```python
tree = Phylo.read(StringIO(newick), "newick")
Phylo.draw_ascii(tree)
```
</details>



## 9) Bonus — format round-trip with `SeqIO`

**Task:**  
Create two `SeqRecord`s, save them to FASTA **and** to GenBank, then read them back. Verify the lengths match.

> Hint: Use `SeqIO.write(records, "file.fasta", "fasta")` and `SeqIO.parse(..., format)`.


In [ ]:

# TODO: Your work here
rec1 = SeqRecord(Seq("ATGGCC"), id="rec1", description="synthetic A")
rec2 = SeqRecord(Seq("ATGAAAAAA"), id="rec2", description="synthetic B")
# Write and read back; compare lengths



<details><summary><strong>View answer</strong></summary>

```python
recs = [rec1, rec2]
SeqIO.write(recs, "tmp.fasta", "fasta")
SeqIO.write(recs, "tmp.gb", "genbank")

lens_fa = [len(r) for r in SeqIO.parse("tmp.fasta", "fasta")]
lens_gb = [len(r) for r in SeqIO.parse("tmp.gb", "genbank")]
print("FASTA lens:", lens_fa)
print("GenBank lens:", lens_gb)
assert lens_fa == lens_gb == [6, 9]
```
</details>



---

**End of notebook.**  
Generated on 2025-10-21 07:05 UTC.
